In [ ]:
import numpy as np
import pandas as pd

from dataclasses import dataclass, asdict
from typing import List, Dict, Any, Optional, Tuple

from scipy import stats
from scipy.signal import welch, hilbert, find_peaks

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.decomposition import PCA
from sklearn.cluster import DBSCAN, OPTICS, AgglomerativeClustering
from sklearn.metrics import silhouette_score
from sklearn.feature_selection import f_classif

import plotly.graph_objects as go

# Optional deps
try:
    import pywt
    HAS_PYWT = True
except Exception:
    HAS_PYWT = False

try:
    import umap
    HAS_UMAP = True
except Exception:
    HAS_UMAP = False

try:
    import hdbscan
    HAS_HDBSCAN = True
except Exception:
    HAS_HDBSCAN = False

try:
    from joblib import Parallel, delayed
    HAS_JOBLIB = True
except Exception:
    HAS_JOBLIB = False


# ----------------------------
# Configs
# ----------------------------
@dataclass
class FeatureConfig:
    fs: float = 62.5e6
    welch_nperseg: int = 1024
    welch_noverlap: Optional[int] = None

    peak_prominence_q: float = 0.90
    min_peak_distance_ms: float = 0.05
    max_echo_peaks: int = 8

    wavelet_name: str = "db4"
    wavelet_levels: int = 6


@dataclass
class ClusterConfig:
    reducer: str = "pca"  # "pca" / "umap"

    # PCA
    pca_variance: float = 0.95

    # UMAP
    umap_n_neighbors: int = 25
    umap_min_dist: float = 0.05
    umap_n_components: int = 2

    # Density clustering
    density_method: str = "hdbscan"  # "hdbscan" / "optics" / "dbscan"

    # HDBSCAN
    hdb_min_cluster_size: int = 20
    hdb_min_samples: Optional[int] = None

    # OPTICS
    optics_min_samples: int = 20
    optics_xi: float = 0.05
    optics_min_cluster_size: float = 0.03

    # DBSCAN
    dbscan_eps: float = 0.6
    dbscan_min_samples: int = 20

    # Optional hierarchical clustering
    do_hierarchical: bool = False
    hier_n_clusters: int = 10
    hier_linkage: str = "ward"


# ----------------------------
# Core feature helpers
# ----------------------------
def robust_norm(x: np.ndarray) -> np.ndarray:
    med = np.median(x)
    mad = np.median(np.abs(x - med))
    if mad < 1e-12:
        s = np.std(x)
        mad = s if s > 1e-12 else 1.0
    return (x - med) / mad


def time_domain_features(x: np.ndarray) -> Dict[str, float]:
    x = np.asarray(x, dtype=np.float64)
    n = len(x)
    if n < 4:
        keys = ["len","mean","std","rms","ptp","abs_mean","abs_std","skew","kurtosis","zcr","crest","impulse","shape","clearance"]
        return {k: np.nan for k in keys}

    mean = float(np.mean(x))
    std = float(np.std(x))
    rms = float(np.sqrt(np.mean(x**2)))
    ptp = float(np.ptp(x))

    absx = np.abs(x)
    abs_mean = float(np.mean(absx))
    abs_std = float(np.std(absx))

    skew = float(stats.skew(x, bias=False))
    kurt = float(stats.kurtosis(x, fisher=True, bias=False))

    zcr = float(np.mean(x[:-1] * x[1:] < 0.0))

    peak = float(np.max(absx))
    crest = peak / (rms + 1e-12)
    impulse = peak / (abs_mean + 1e-12)
    shape = rms / (abs_mean + 1e-12)
    clearance = peak / ((np.mean(np.sqrt(absx)) + 1e-12) ** 2)

    return {
        "len": float(n),
        "mean": mean,
        "std": std,
        "rms": rms,
        "ptp": ptp,
        "abs_mean": abs_mean,
        "abs_std": abs_std,
        "skew": skew,
        "kurtosis": kurt,
        "zcr": zcr,
        "crest": float(crest),
        "impulse": float(impulse),
        "shape": float(shape),
        "clearance": float(clearance),
    }


def echo_peak_features(x: np.ndarray, cfg: FeatureConfig) -> Dict[str, float]:
    x = np.asarray(x, dtype=np.float64)
    n = len(x)
    if n < 8:
        keys = [
            "env_peak_count","env_top1_amp","env_top2_amp",
            "env_top1_top2_ratio","env_top1_top2_dt_ms",
            "env_peak_amp_mean","env_peak_amp_std"
        ]
        return {k: np.nan for k in keys}

    env = np.abs(hilbert(x))
    q = np.quantile(env, cfg.peak_prominence_q)
    min_dist = max(1, int((cfg.min_peak_distance_ms / 1000.0) * cfg.fs))

    peaks, props = find_peaks(env, height=q, distance=min_dist)

    if len(peaks) == 0:
        return {
            "env_peak_count": 0.0,
            "env_top1_amp": 0.0,
            "env_top2_amp": 0.0,
            "env_top1_top2_ratio": np.nan,
            "env_top1_top2_dt_ms": np.nan,
            "env_peak_amp_mean": 0.0,
            "env_peak_amp_std": 0.0,
        }

    amps = props["peak_heights"]
    order = np.argsort(amps)[::-1]
    amps_sorted = amps[order]
    peaks_sorted = peaks[order]

    top1_amp = float(amps_sorted[0])
    top2_amp = float(amps_sorted[1]) if len(amps_sorted) > 1 else 0.0
    ratio = (top1_amp / (top2_amp + 1e-12)) if top2_amp > 0 else np.nan

    if len(peaks_sorted) > 1:
        dt = abs(int(peaks_sorted[0]) - int(peaks_sorted[1])) / cfg.fs * 1000.0
    else:
        dt = np.nan

    amps_cap = amps_sorted[:cfg.max_echo_peaks]
    return {
        "env_peak_count": float(len(peaks)),
        "env_top1_amp": top1_amp,
        "env_top2_amp": top2_amp,
        "env_top1_top2_ratio": float(ratio) if np.isfinite(ratio) else np.nan,
        "env_top1_top2_dt_ms": float(dt) if np.isfinite(dt) else np.nan,
        "env_peak_amp_mean": float(np.mean(amps_cap)),
        "env_peak_amp_std": float(np.std(amps_cap)),
    }


def spectral_features(x: np.ndarray, cfg: FeatureConfig) -> Dict[str, float]:
    x = np.asarray(x, dtype=np.float64)
    n = len(x)
    if n < 16:
        keys = ["spec_centroid","spec_bw","spec_entropy","spec_flatness",
                "bp_0_0.1","bp_0.1_0.2","bp_0.2_0.4","bp_0.4_0.6","bp_0.6_0.8","bp_0.8_1"]
        return {k: np.nan for k in keys}

    nperseg = min(cfg.welch_nperseg, n)
    noverlap = cfg.welch_noverlap if cfg.welch_noverlap is not None else nperseg // 2
    noverlap = min(noverlap, nperseg - 1)

    f, Pxx = welch(x, fs=cfg.fs, nperseg=nperseg, noverlap=noverlap, detrend="constant", scaling="density")
    Pxx = np.maximum(Pxx, 1e-20)

    p = Pxx / np.sum(Pxx)
    centroid = float(np.sum(f * p))
    bw = float(np.sqrt(np.sum(((f - centroid) ** 2) * p)))
    entropy = float(-np.sum(p * np.log(p + 1e-20)))
    flatness = float(np.exp(np.mean(np.log(Pxx))) / (np.mean(Pxx) + 1e-20))

    nyq = cfg.fs / 2.0
    bands = [(0.0,0.1),(0.1,0.2),(0.2,0.4),(0.4,0.6),(0.6,0.8),(0.8,1.0)]
    total_power = float(np.trapz(Pxx, f))

    out = {
        "spec_centroid": centroid,
        "spec_bw": bw,
        "spec_entropy": entropy,
        "spec_flatness": flatness,
    }
    for (a, b) in bands:
        fmin, fmax = a * nyq, b * nyq
        mask = (f >= fmin) & (f <= fmax)
        bp = float(np.trapz(Pxx[mask], f[mask])) if np.any(mask) else 0.0
        key = f"bp_{a:g}_{b:g}"
        out[key] = bp / (total_power + 1e-20)

    return out


def wavelet_features(x: np.ndarray, cfg: FeatureConfig) -> Dict[str, float]:
    out = {f"w_energy_L{i}": np.nan for i in range(1, cfg.wavelet_levels + 1)}
    out["w_energy_A"] = np.nan
    out["w_energy_entropy"] = np.nan

    if not HAS_PYWT:
        return out

    x = np.asarray(x, dtype=np.float64)
    n = len(x)
    if n < 16:
        return out

    wavelet = pywt.Wavelet(cfg.wavelet_name)
    max_level = pywt.dwt_max_level(data_len=n, filter_len=wavelet.dec_len)
    level = min(cfg.wavelet_levels, max_level) if max_level >= 1 else 1

    coeffs = pywt.wavedec(x, wavelet=cfg.wavelet_name, level=level)
    cA = coeffs[0]
    details = coeffs[1:]  # cD_level ... cD1

    energies = [float(np.sum(np.square(cD))) for cD in details]
    eA = float(np.sum(np.square(cA)))

    energies_L = list(reversed(energies))  # L1..Llevel
    if len(energies_L) < cfg.wavelet_levels:
        energies_L += [0.0] * (cfg.wavelet_levels - len(energies_L))
    energies_L = energies_L[:cfg.wavelet_levels]

    total = eA + sum(energies_L) + 1e-20
    energies_norm = [e / total for e in energies_L]
    eA_norm = eA / total

    p = np.array(energies_norm + [eA_norm], dtype=np.float64)
    p = p / (p.sum() + 1e-20)
    wentropy = float(-np.sum(p * np.log(p + 1e-20)))

    for i, v in enumerate(energies_norm, start=1):
        out[f"w_energy_L{i}"] = float(v)
    out["w_energy_A"] = float(eA_norm)
    out["w_energy_entropy"] = wentropy
    return out


def extract_features_one(x: np.ndarray, cfg: FeatureConfig) -> Dict[str, float]:
    x = np.asarray(x, dtype=np.float64)
    if len(x) == 0:
        return {"len": 0.0}

    feats = {}
    # raw (amplitude + shape)
    feats.update({f"raw_{k}": v for k, v in time_domain_features(x).items()})
    feats.update({f"raw_{k}": v for k, v in echo_peak_features(x, cfg).items()})
    feats.update({f"raw_{k}": v for k, v in spectral_features(x, cfg).items()})
    feats.update({f"raw_{k}": v for k, v in wavelet_features(x, cfg).items()})

    # robust-normalized (more shape-oriented)
    xn = robust_norm(x)
    feats.update({f"norm_{k}": v for k, v in time_domain_features(xn).items()})
    feats.update({f"norm_{k}": v for k, v in echo_peak_features(xn, cfg).items()})
    feats.update({f"norm_{k}": v for k, v in spectral_features(xn, cfg).items()})
    feats.update({f"norm_{k}": v for k, v in wavelet_features(xn, cfg).items()})
    return feats


def extract_features(signals: List[np.ndarray], cfg: FeatureConfig, n_jobs: int = 1) -> pd.DataFrame:
    if HAS_JOBLIB and n_jobs != 1:
        rows = Parallel(n_jobs=n_jobs, prefer="processes")(
            delayed(extract_features_one)(x, cfg) for x in signals
        )
    else:
        rows = [extract_features_one(x, cfg) for x in signals]
    return pd.DataFrame(rows)


# ----------------------------
# Preprocess feature matrix
# ----------------------------
def build_feature_matrix(feat_df: pd.DataFrame) -> Tuple[np.ndarray, List[str], SimpleImputer, RobustScaler]:
    feature_cols = feat_df.columns.tolist()
    Xraw = feat_df.values

    imputer = SimpleImputer(strategy="median")
    Ximp = imputer.fit_transform(Xraw)

    scaler = RobustScaler(with_centering=True, with_scaling=True)
    X = scaler.fit_transform(Ximp)
    return X, feature_cols, imputer, scaler


# ----------------------------
# Dim reduction
# ----------------------------
def reduce_for_clustering(X: np.ndarray, ccfg: ClusterConfig, random_state: int = 42) -> Tuple[np.ndarray, Dict[str, Any]]:
    info = {}
    if ccfg.reducer.lower() == "umap":
        if not HAS_UMAP:
            ccfg = ClusterConfig(**{**asdict(ccfg), "reducer": "pca"})  # fallback

    if ccfg.reducer.lower() == "umap" and HAS_UMAP:
        reducer = umap.UMAP(
            n_neighbors=ccfg.umap_n_neighbors,
            min_dist=ccfg.umap_min_dist,
            n_components=ccfg.umap_n_components,
            random_state=random_state,
        )
        Z = reducer.fit_transform(X)
        info["reducer"] = "umap"
        info["umap_params"] = {
            "n_neighbors": ccfg.umap_n_neighbors,
            "min_dist": ccfg.umap_min_dist,
            "n_components": ccfg.umap_n_components,
        }
        return Z, info

    pca = PCA(n_components=ccfg.pca_variance, random_state=random_state)
    Z = pca.fit_transform(X)
    info["reducer"] = "pca"
    info["pca_n_components"] = int(Z.shape[1])
    info["pca_explained_variance_sum"] = float(np.sum(pca.explained_variance_ratio_))
    return Z, info


def reduce_for_viz_2d(X: np.ndarray, ccfg: ClusterConfig, random_state: int = 42) -> np.ndarray:
    # always output 2D for plotly
    if ccfg.reducer.lower() == "umap" and HAS_UMAP:
        reducer2 = umap.UMAP(
            n_neighbors=ccfg.umap_n_neighbors,
            min_dist=ccfg.umap_min_dist,
            n_components=2,
            random_state=random_state,
        )
        return reducer2.fit_transform(X)

    pca2 = PCA(n_components=2, random_state=random_state)
    return pca2.fit_transform(X)


# ----------------------------
# Clustering
# ----------------------------
def density_cluster(Z: np.ndarray, ccfg: ClusterConfig) -> Tuple[np.ndarray, Dict[str, Any]]:
    method = ccfg.density_method.lower()
    info = {"density_method": method}

    if method == "hdbscan":
        if HAS_HDBSCAN:
            clusterer = hdbscan.HDBSCAN(
                min_cluster_size=ccfg.hdb_min_cluster_size,
                min_samples=ccfg.hdb_min_samples,
            )
            labels = clusterer.fit_predict(Z)
            info["hdbscan_params"] = {
                "min_cluster_size": ccfg.hdb_min_cluster_size,
                "min_samples": ccfg.hdb_min_samples,
            }
            if hasattr(clusterer, "probabilities_"):
                info["hdbscan_prob_mean"] = float(np.mean(clusterer.probabilities_))
            return labels, info
        # fallback
        method = "optics"
        info["density_method"] = "optics"

    if method == "optics":
        clusterer = OPTICS(
            min_samples=ccfg.optics_min_samples,
            xi=ccfg.optics_xi,
            min_cluster_size=ccfg.optics_min_cluster_size,
        )
        labels = clusterer.fit_predict(Z)
        info["optics_params"] = {
            "min_samples": ccfg.optics_min_samples,
            "xi": ccfg.optics_xi,
            "min_cluster_size": ccfg.optics_min_cluster_size,
        }
        return labels, info

    if method == "dbscan":
        clusterer = DBSCAN(eps=ccfg.dbscan_eps, min_samples=ccfg.dbscan_min_samples)
        labels = clusterer.fit_predict(Z)
        info["dbscan_params"] = {"eps": ccfg.dbscan_eps, "min_samples": ccfg.dbscan_min_samples}
        return labels, info

    raise ValueError(f"Unknown density_method: {ccfg.density_method}")


def hierarchical_cluster(Z: np.ndarray, ccfg: ClusterConfig) -> np.ndarray:
    clusterer = AgglomerativeClustering(n_clusters=ccfg.hier_n_clusters, linkage=ccfg.hier_linkage)
    return clusterer.fit_predict(Z)


# ----------------------------
# Cluster utilities
# ----------------------------
def list_clusters(labels: np.ndarray, include_noise: bool = False) -> List[int]:
    u = sorted(set(labels.tolist()))
    if not include_noise and -1 in u:
        u.remove(-1)
    return u


def medoid_index(Z: np.ndarray, idxs: np.ndarray, max_n: int = 600, seed: int = 42) -> int:
    ids = np.asarray(idxs)
    if len(ids) == 1:
        return int(ids[0])

    if len(ids) > max_n:
        rng = np.random.default_rng(seed)
        ids_sub = rng.choice(ids, size=max_n, replace=False)
    else:
        ids_sub = ids

    V = Z[ids_sub]
    G = V @ V.T
    sq = np.sum(V**2, axis=1, keepdims=True)
    D2 = np.maximum(sq + sq.T - 2.0 * G, 0.0)
    D = np.sqrt(D2)
    score = np.mean(D, axis=1)
    best = ids_sub[np.argmin(score)]
    return int(best)


def cluster_summary(feat_df: pd.DataFrame, labels: np.ndarray) -> pd.DataFrame:
    df = feat_df.copy()
    df["cluster"] = labels

    # a readable subset (exists if you used this feature set)
    preferred = [
        "norm_env_peak_count",
        "norm_env_top1_top2_dt_ms",
        "norm_spec_centroid",
        "norm_spec_entropy",
        "norm_w_energy_entropy",
        "raw_rms",
        "raw_ptp",
    ]
    cols = [c for c in preferred if c in df.columns]
    if len(cols) == 0:
        cols = df.select_dtypes(include=[np.number]).columns.tolist()
        cols = [c for c in cols if c != "cluster"][:12]

    out = df.groupby("cluster")[cols].agg(["count", "mean", "std"])
    return out


def feature_anova_top(X: np.ndarray, labels: np.ndarray, feature_names: List[str], top_k: int = 30) -> pd.DataFrame:
    m = labels != -1
    y = labels[m]
    X2 = X[m]
    if len(np.unique(y)) < 2:
        return pd.DataFrame(columns=["feature", "F", "p"])

    F, p = f_classif(X2, y)
    df = pd.DataFrame({"feature": feature_names, "F": F, "p": p})
    df = df.replace([np.inf, -np.inf], np.nan).dropna()
    df = df.sort_values("F", ascending=False).head(top_k)
    return df


# ----------------------------
# Plotly visualizations
# ----------------------------
def plot_embedding_plotly(
    Z2: np.ndarray,
    labels: np.ndarray,
    ids: Optional[List[Any]] = None,
    extra_hover: Optional[Dict[str, np.ndarray]] = None,
    title: str = "2D embedding (colored by cluster)",
) -> go.Figure:
    ids = ids if ids is not None else list(range(len(labels)))
    extra_hover = extra_hover or {} 

    hover_text = []
    for i in range(len(labels)):
        parts = [f"id={ids[i]}", f"cluster={labels[i]}"]
        for k, arr in extra_hover.items():
            parts.append(f"{k}={arr[i]}")
        hover_text.append("<br>".join(parts))

    fig = go.Figure()
    for c in sorted(set(labels.tolist())):
        mask = labels == c
        name = "noise(-1)" if c == -1 else f"cluster {c}"
        fig.add_trace(go.Scattergl(
            x=Z2[mask, 0],
            y=Z2[mask, 1],
            mode="markers",
            name=name,
            marker=dict(size=6),
            text=np.array(hover_text, dtype=object)[mask],
            hoverinfo="text",
        ))

    fig.update_layout(
        title=title,
        xaxis_title="dim-1",
        yaxis_title="dim-2",
        legend_title="labels",
        height=650,
    )
    return fig


def plot_cluster_waveforms_plotly(
    signals: List[np.ndarray],
    labels: np.ndarray,
    fs: float,
    cluster_id: int,
    Z_cluster_space: Optional[np.ndarray] = None,
    n_samples: int = 10,
    align_by_peak: bool = True,
    seed: int = 42,
) -> go.Figure:
    idxs = np.where(labels == cluster_id)[0]
    if len(idxs) == 0:
        raise ValueError(f"No samples for cluster {cluster_id}")

    rng = np.random.default_rng(seed)
    pick = idxs if len(idxs) <= n_samples else rng.choice(idxs, size=n_samples, replace=False)

    fig = go.Figure()

    # overlay random samples
    for i in pick:
        x = np.asarray(signals[int(i)], dtype=np.float64)
        if len(x) < 2:
            continue
        if align_by_peak:
            p = int(np.argmax(np.abs(x)))
            t = (np.arange(len(x)) - p) / fs * 1e3
        else:
            t = np.arange(len(x)) / fs * 1e3
        fig.add_trace(go.Scatter(
            x=t, y=x, mode="lines",
            name=f"sample {int(i)}",
            opacity=0.25,
            hoverinfo="name",
            showlegend=False,
        ))

    # medoid highlight (optional)
    if Z_cluster_space is not None:
        med = medoid_index(Z_cluster_space, idxs)
        xm = np.asarray(signals[int(med)], dtype=np.float64)
        pm = int(np.argmax(np.abs(xm)))
        tm = (np.arange(len(xm)) - pm) / fs * 1e3 if align_by_peak else np.arange(len(xm)) / fs * 1e3
        fig.add_trace(go.Scatter(
            x=tm, y=xm, mode="lines",
            name=f"medoid {med}",
            line=dict(width=3),
        ))

    fig.update_layout(
        title=f"Cluster {cluster_id}: waveforms (n={len(idxs)}), {'peak-aligned' if align_by_peak else 'raw time'}",
        xaxis_title="Time (ms)",
        yaxis_title="Amplitude",
        height=520,
    )
    return fig


def mean_psd_for_indices(
    signals: List[np.ndarray],
    idxs: np.ndarray,
    fs: float,
    nperseg: int = 1024,
    max_samples: int = 250,
    seed: int = 42,
) -> Tuple[np.ndarray, np.ndarray, int]:
    rng = np.random.default_rng(seed)
    if len(idxs) > max_samples:
        idxs = rng.choice(idxs, size=max_samples, replace=False)

    f_ref = None
    psd_list = []
    used = 0

    for i in idxs:
        x = np.asarray(signals[int(i)], dtype=np.float64)
        if len(x) < 16:
            continue
        seg = min(nperseg, len(x))
        ov = min(seg // 2, seg - 1)
        f, Pxx = welch(x, fs=fs, nperseg=seg, noverlap=ov, detrend="constant", scaling="density")
        if f_ref is None:
            f_ref = f
            psd_list.append(Pxx)
        else:
            psd_list.append(np.interp(f_ref, f, Pxx))
        used += 1

    if f_ref is None or used == 0:
        raise ValueError("No valid PSD computed (signals too short?)")

    P = np.vstack(psd_list)
    Pm = np.mean(P, axis=0)
    return f_ref, Pm, used


def plot_cluster_psd_plotly(
    signals: List[np.ndarray],
    labels: np.ndarray,
    fs: float,
    cluster_id: int,
    nperseg: int = 1024,
    max_samples: int = 250,
) -> go.Figure:
    idxs = np.where(labels == cluster_id)[0]
    f, Pm, used = mean_psd_for_indices(signals, idxs, fs, nperseg=nperseg, max_samples=max_samples)

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=f, y=Pm + 1e-20, mode="lines", name=f"cluster {cluster_id} mean PSD"))
    fig.update_layout(
        title=f"Cluster {cluster_id}: mean PSD (Welch), used={used}",
        xaxis_title="Frequency (Hz)",
        yaxis_title="PSD",
        yaxis_type="log",
        height=520,
    )
    return fig


def plot_cluster_wavelet_energy_plotly(
    feat_df: pd.DataFrame,
    labels: np.ndarray,
    cluster_id: int,
    prefix: str = "norm_w_energy_L",
) -> go.Figure:
    cols = [c for c in feat_df.columns if c.startswith(prefix)]
    if len(cols) == 0:
        raise ValueError("No wavelet energy columns found. (PyWavelets installed? wavelet features enabled?)")

    idxs = np.where(labels == cluster_id)[0]
    if len(idxs) == 0:
        raise ValueError(f"No samples for cluster {cluster_id}")

    m = feat_df.iloc[idxs][cols].mean(axis=0)

    fig = go.Figure()
    fig.add_trace(go.Bar(x=cols, y=m.values, name="mean wavelet energies"))
    fig.update_layout(
        title=f"Cluster {cluster_id}: mean normalized wavelet detail energies",
        xaxis_title="Wavelet detail levels",
        yaxis_title="Energy (normalized)",
        height=420,
    )
    return fig


# ----------------------------
# Pipeline runner (Notebook entry)
# ----------------------------
def run_routeA(
    signals: List[np.ndarray],
    ids: Optional[List[Any]],
    fs: float,
    feature_cfg: Optional[FeatureConfig] = None,
    cluster_cfg: Optional[ClusterConfig] = None,
    n_jobs: int = 1,
    random_state: int = 42,
) -> Dict[str, Any]:
    if ids is None:
        ids = list(range(len(signals)))
    if len(ids) != len(signals):
        raise ValueError("ids length must match signals length")

    feature_cfg = feature_cfg or FeatureConfig(fs=fs)
    cluster_cfg = cluster_cfg or ClusterConfig()

    # 1) feature extraction
    feat_df = extract_features(signals, feature_cfg, n_jobs=n_jobs)

    # 2) build matrix (impute + robust scale)
    X, feature_names, imputer, scaler = build_feature_matrix(feat_df)

    # 3) reduce for clustering + reduce for viz
    Z = reduce_for_clustering(X, cluster_cfg, random_state=random_state)[0]
    Z2 = reduce_for_viz_2d(X, cluster_cfg, random_state=random_state)

    # 4) cluster (density)
    labels, cl_info = density_cluster(Z, cluster_cfg)

    # 5) optional hierarchical
    labels_h = None
    if cluster_cfg.do_hierarchical:
        labels_h = hierarchical_cluster(Z, cluster_cfg)

    # 6) metrics
    metrics = {}
    m = labels != -1
    if np.sum(m) > 10 and len(np.unique(labels[m])) >= 2:
        metrics["silhouette_non_noise"] = float(silhouette_score(Z[m], labels[m]))

    # 7) summary + anova
    summary_df = cluster_summary(feat_df, labels)
    anova_df = feature_anova_top(X, labels, feature_names, top_k=40)

    return {
        "ids": ids,
        "feature_cfg": feature_cfg,
        "cluster_cfg": cluster_cfg,
        "feat_df": feat_df,
        "X": X,
        "feature_names": feature_names,
        "imputer": imputer,
        "scaler": scaler,
        "Z": Z,          # clustering space
        "Z2": Z2,        # 2D for visualization
        "labels": labels,
        "labels_hier": labels_h,
        "cluster_info": cl_info,
        "metrics": metrics,
        "summary_df": summary_df,
        "anova_df": anova_df,
    }


In [ ]:
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio

# ---------- 小工具：对齐 + 固定窗口 + 重采样（仅用于可视化） ----------
def _extract_peak_aligned_window(
    x: np.ndarray,
    fs: float,
    pre_ms: float,
    post_ms: float,
) -> np.ndarray:
    """
    以 |x| 最大点为中心，截取固定时长窗口（pre_ms + post_ms），不足则 0-padding。
    返回：np.ndarray, shape (win_len_samples,)
    """
    x = np.asarray(x, dtype=np.float64)
    if x.size == 0:
        win_len = max(2, int((pre_ms + post_ms) / 1000.0 * fs))
        return np.zeros(win_len, dtype=np.float64)

    p = int(np.argmax(np.abs(x)))
    pre = int(pre_ms / 1000.0 * fs)
    post = int(post_ms / 1000.0 * fs)
    win_len = pre + post
    win_len = max(2, win_len)

    start = p - pre
    end = p + post

    # 取有效区间
    s0 = max(0, start)
    e0 = min(len(x), end)
    seg = x[s0:e0]

    # padding 到固定长度
    out = np.zeros(win_len, dtype=np.float64)
    left_pad = s0 - start  # start<0 时为正
    out[left_pad:left_pad + len(seg)] = seg
    return out


def _resample_1d(y: np.ndarray, target_len: int) -> np.ndarray:
    """
    线性插值重采样到 target_len（仅用于可视化对齐长度一致）。
    """
    y = np.asarray(y, dtype=np.float64)
    if y.size <= 1:
        return np.zeros(target_len, dtype=np.float64)
    x_old = np.linspace(0.0, 1.0, y.size)
    x_new = np.linspace(0.0, 1.0, target_len)
    return np.interp(x_new, x_old, y)


def _aligned_resampled_matrix(
    signals,
    idxs,
    fs,
    pre_ms,
    post_ms,
    viz_len,
    max_samples=None,
    seed=42,
):
    """
    对簇内样本：peak-align -> fixed window -> resample -> 得到矩阵 (N_used, viz_len)
    若 max_samples 不为 None 且簇太大，则随机抽样用于统计（减少内存/时间）。
    """
    idxs = np.asarray(idxs, dtype=int)
    if max_samples is not None and len(idxs) > max_samples:
        rng = np.random.default_rng(seed)
        idxs = rng.choice(idxs, size=max_samples, replace=False)

    M = np.zeros((len(idxs), viz_len), dtype=np.float64)
    for r, i in enumerate(idxs):
        w = _extract_peak_aligned_window(signals[i], fs, pre_ms, post_ms)
        M[r] = _resample_1d(w, viz_len)
    return M, idxs


# ---------- 质心：嵌入空间 centroid + 波形 centroid ----------
def _cluster_centroids_2d(Z2, labels, include_noise=False):
    """
    返回每簇在 2D embedding 上的质心坐标 + 样本数
    """
    labs = sorted(set(labels.tolist()))
    if not include_noise and -1 in labs:
        labs.remove(-1)

    cx, cy, ns = [], [], []
    for c in labs:
        m = labels == c
        cx.append(float(np.mean(Z2[m, 0])))
        cy.append(float(np.mean(Z2[m, 1])))
        ns.append(int(np.sum(m)))
    return labs, np.array(cx), np.array(cy), np.array(ns)


# ---------- Embedding：点云 + centroid ----------
def build_embedding_with_centroids_fig(res, show_noise=True):
    Z2 = res["Z2"]
    labels = res["labels"]
    ids = res["ids"]

    fig = go.Figure()

    # 点云（按簇）
    for c in sorted(set(labels.tolist())):
        if (not show_noise) and c == -1:
            continue
        m = labels == c
        name = "noise(-1)" if c == -1 else f"cluster {c}"
        fig.add_trace(go.Scattergl(
            x=Z2[m, 0], y=Z2[m, 1],
            mode="markers",
            name=name,
            marker=dict(size=5),
            text=[f"id={ids[i]}<br>cluster={labels[i]}" for i in np.where(m)[0]],
            hoverinfo="text",
        ))

    # centroid
    labs, cx, cy, ns = _cluster_centroids_2d(Z2, labels, include_noise=show_noise)
    fig.add_trace(go.Scatter(
        x=cx, y=cy,
        mode="markers+text",
        name="centroids",
        marker=dict(size=14, symbol="x"),
        text=[f"C{c} (n={n})" for c, n in zip(labs, ns)],
        textposition="top center",
        hoverinfo="text",
    ))

    fig.update_layout(
        title="Embedding (points) + cluster centroids",
        xaxis_title="dim-1",
        yaxis_title="dim-2",
        height=680,
    )
    return fig


def build_centroid_only_fig(res, show_noise=False):
    Z2 = res["Z2"]
    labels = res["labels"]
    labs, cx, cy, ns = _cluster_centroids_2d(Z2, labels, include_noise=show_noise)

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=cx, y=cy,
        mode="markers+text",
        marker=dict(size=np.clip(ns / np.max(ns) * 30, 10, 30), symbol="x"),
        text=[f"C{c}" for c in labs],
        textposition="top center",
        hovertext=[f"cluster={c}<br>n={n}<br>centroid=({x:.3f},{y:.3f})" for c, n, x, y in zip(labs, ns, cx, cy)],
        hoverinfo="text",
        name="centroids",
    ))
    fig.update_layout(
        title="Cluster centroids only (marker size ~ cluster size)",
        xaxis_title="dim-1",
        yaxis_title="dim-2",
        height=560,
    )
    return fig


# ---------- Waveform 浏览器：下拉选簇 + hover 高亮 ----------
def build_waveform_browser_fig(
    res,
    signals,
    fs,
    pre_ms=0.02,        # 20 µs：默认用于 PD 脉冲可视化（你可按实际改）
    post_ms=0.10,       # 100 µs
    viz_len=1024,       # 重采样长度（越大越清晰但 HTML 越大）
    display_samples=25, # 每簇显示多少条真实波形（剩下用统计带表达）
    max_quantile_samples=2000,  # 计算分位带最多采样这么多条（防止巨簇内存爆）
    seed=42,
    show_noise=False,
):
    labels = res["labels"]
    ids = res["ids"]
    Z = res.get("Z", None)   # 作为 medoid 的空间（如果没有也能跑）

    clusters = sorted(set(labels.tolist()))
    if not show_noise and -1 in clusters:
        clusters.remove(-1)
    if len(clusters) == 0:
        raise ValueError("No clusters found (after noise filtering).")

    # 每个 cluster 一组 traces，做 dropdown 切换可见性
    fig = go.Figure()
    trace_ranges = []  # [(start_idx, end_idx_exclusive), ...] per cluster

    t_ms = np.linspace(-pre_ms, post_ms, viz_len)  # 统一时间轴（ms）

    rng = np.random.default_rng(seed)

    def _medoid_idx_for_cluster(c):
        # 如果你 res 里有 Z（聚类空间），这里可用“最居中样本”作为代表
        if Z is None:
            return None
        idxs = np.where(labels == c)[0]
        if len(idxs) == 0:
            return None
        # 简化版 medoid：用 cluster centroid 最近点（O(k)），避免 O(k^2)
        C = np.mean(Z[idxs], axis=0)
        d = np.sum((Z[idxs] - C) ** 2, axis=1)
        return int(idxs[np.argmin(d)])

    for ci, c in enumerate(clusters):
        start = len(fig.data)
        idxs = np.where(labels == c)[0]
        n = len(idxs)

        # 1) 用于统计带的矩阵（可能抽样）
        M, idxs_used_q = _aligned_resampled_matrix(
            signals, idxs, fs, pre_ms, post_ms, viz_len,
            max_samples=max_quantile_samples, seed=seed
        )
        q10 = np.quantile(M, 0.10, axis=0)
        q90 = np.quantile(M, 0.90, axis=0)
        med = np.median(M, axis=0)
        mean = np.mean(M, axis=0)  # 波形质心（mean waveform）

        # 2) 显示用的少量真实样本（单独 trace，便于 hover 高亮）
        if n <= display_samples:
            pick = idxs
        else:
            pick = rng.choice(idxs, size=display_samples, replace=False)

        # 3) medoid waveform（如果可计算）
        medoid = _medoid_idx_for_cluster(c)
        if medoid is not None:
            w_medoid = _extract_peak_aligned_window(signals[medoid], fs, pre_ms, post_ms)
            w_medoid = _resample_1d(w_medoid, viz_len)

        # --- 添加 traces（注意：每个 cluster 的 traces 初始可见性由 dropdown 控制） ---

        # band upper
        fig.add_trace(go.Scatter(
            x=t_ms, y=q90, mode="lines",
            line=dict(width=0),
            hoverinfo="skip",
            showlegend=False,
            name=f"C{c} q90",
            visible=(ci == 0),
        ))
        # band lower (fill)
        fig.add_trace(go.Scatter(
            x=t_ms, y=q10, mode="lines",
            fill="tonexty",
            line=dict(width=0),
            hoverinfo="skip",
            showlegend=False,
            name=f"C{c} q10-q90",
            visible=(ci == 0),
            opacity=0.25,
        ))

        # median
        fig.add_trace(go.Scatter(
            x=t_ms, y=med, mode="lines",
            name=f"C{c} median",
            line=dict(width=2),
            visible=(ci == 0),
        ))

        # centroid waveform (mean)
        fig.add_trace(go.Scatter(
            x=t_ms, y=mean, mode="lines",
            name=f"C{c} centroid(mean)",
            line=dict(width=3, dash="dash"),
            visible=(ci == 0),
        ))

        # medoid waveform
        if medoid is not None:
            fig.add_trace(go.Scatter(
                x=t_ms, y=w_medoid, mode="lines",
                name=f"C{c} medoid(id={ids[medoid]})",
                line=dict(width=4),
                visible=(ci == 0),
            ))

        # sample waveforms (meta='sample' 用于 hover 高亮识别)
        for i in pick:
            w = _extract_peak_aligned_window(signals[int(i)], fs, pre_ms, post_ms)
            w = _resample_1d(w, viz_len)
            fig.add_trace(go.Scatter(
                x=t_ms, y=w, mode="lines",
                name=f"C{c} sample",
                meta="sample",
                opacity=0.25,
                line=dict(width=1),
                showlegend=False,
                visible=(ci == 0),
                hovertext=f"id={ids[int(i)]}<br>cluster={c}<br>sample_index={int(i)}<br>n_in_cluster={n}",
                hoverinfo="text",
            ))

        end = len(fig.data)
        trace_ranges.append((start, end))

    # dropdown：切换 cluster 可见性
    buttons = []
    total_traces = len(fig.data)

    for ci, c in enumerate(clusters):
        vis = [False] * total_traces
        s, e = trace_ranges[ci]
        for k in range(s, e):
            vis[k] = True

        buttons.append(dict(
            label=f"Cluster {c}",
            method="update",
            args=[
                {"visible": vis},
                {"title": f"Waveform browser — Cluster {c} (peak-aligned, q10–q90 band + median + centroid + samples)"}
            ],
        ))

    fig.update_layout(
        title=f"Waveform browser — Cluster {clusters[0]} (peak-aligned, q10–q90 band + median + centroid + samples)",
        xaxis_title="Time (ms, peak aligned)",
        yaxis_title="Amplitude",
        height=620,
        updatemenus=[dict(
            buttons=buttons,
            direction="down",
            showactive=True,
            x=1.02, xanchor="left",
            y=1.0, yanchor="top",
        )],
        margin=dict(r=260),  # 给右侧 dropdown 留空间
    )

    return fig


# ---------- 写 HTML：组合多个 plotly figure + 注入 hover 高亮 JS ----------
def export_cluster_report_html(
    res,
    signals,
    fs,
    out_html="cluster_report.html",
    show_noise_in_embedding=True,
    show_noise_in_waveforms=False,
    pre_ms=0.02,
    post_ms=0.10,
    viz_len=1024,
    display_samples=25,
    max_quantile_samples=2000,
    offline_embed_plotlyjs=False,
):
    # 1) Figures
    fig_embed = build_embedding_with_centroids_fig(res, show_noise=show_noise_in_embedding)
    fig_centroids = build_centroid_only_fig(res, show_noise=False)
    fig_wave = build_waveform_browser_fig(
        res, signals, fs,
        pre_ms=pre_ms, post_ms=post_ms, viz_len=viz_len,
        display_samples=display_samples,
        max_quantile_samples=max_quantile_samples,
        show_noise=show_noise_in_waveforms,
    )

    # 2) hover 高亮 JS（只对 meta='sample' 的 trace 生效）
    hover_js = r"""
    var gd = document.getElementById('{plot_id}');
    var prev = null;

    function isSampleTrace(pt){
        try { return (pt && pt.data && pt.data.meta === 'sample'); } catch(e){ return false; }
    }

    gd.on('plotly_hover', function(evt){
        if(!evt || !evt.points || evt.points.length === 0) return;
        var pt = evt.points[0];
        if(!isSampleTrace(pt)) return;

        var cn = pt.curveNumber;

        if(prev !== null && prev !== cn){
            Plotly.restyle(gd, {'line.width': 1, 'opacity': 0.25}, [prev]);
        }
        Plotly.restyle(gd, {'line.width': 4, 'opacity': 1.0}, [cn]);
        prev = cn;
    });

    gd.on('plotly_unhover', function(evt){
        if(!evt || !evt.points || evt.points.length === 0) return;
        var pt = evt.points[0];
        if(!isSampleTrace(pt)) return;

        var cn = pt.curveNumber;
        Plotly.restyle(gd, {'line.width': 1, 'opacity': 0.25}, [cn]);
        prev = null;
    });
    """

    # 3) HTML fragments（PlotlyJS 只加载一次）
    include_js = True if offline_embed_plotlyjs else "cdn"

    html_embed = pio.to_html(
        fig_embed,
        include_plotlyjs=include_js,
        full_html=False,
        div_id="fig_embedding"
    )

    html_centroids = pio.to_html(
        fig_centroids,
        include_plotlyjs=False,
        full_html=False,
        div_id="fig_centroids_only"
    )

    html_wave = pio.to_html(
        fig_wave,
        include_plotlyjs=False,
        full_html=False,
        div_id="fig_waveform_browser",
        post_script=hover_js,  # 注入 hover 高亮逻辑
    )

    # 4) 合成总 HTML
    page = f"""
<!doctype html>
<html>
<head>
  <meta charset="utf-8" />
  <title>PD Clustering Report</title>
  <style>
    body {{ font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, "Helvetica Neue", Arial; margin: 18px; }}
    h2 {{ margin-top: 34px; }}
    .note {{ color: #444; line-height: 1.5; max-width: 1000px; }}
    .block {{ margin-top: 10px; }}
  </style>
</head>
<body>
  <h1>PD Clustering Report (Plotly)</h1>
  <div class="note">
    <b>交互说明</b>：<br/>
    1) Embedding 图用于看簇之间的分离度与整体结构；标记 “x” 为每簇质心（2D 均值）。<br/>
    2) Waveform Browser：用右侧下拉菜单切换簇。图中包含 q10–q90 统计带、median、centroid(mean waveform)、medoid，以及少量真实样本线。<br/>
    3) 将鼠标移动到某条样本线（细线）上，会自动高亮该线，便于在密集区域定位。<br/>
  </div>

  <h2>1) Embedding + Centroids</h2>
  <div class="block">{html_embed}</div>

  <h2>2) Centroids Only</h2>
  <div class="block">{html_centroids}</div>

  <h2>3) Waveform Browser (hover highlight)</h2>
  <div class="block">{html_wave}</div>

</body>
</html>
"""
    with open(out_html, "w", encoding="utf-8") as f:
        f.write(page)

    return out_html
